# g1_limpo — treino DO ZERO (`zero01`)

Começa uma linhagem NOVA: sem checkpoint de entrada, sem dataset de entrada, sem
retomada. A sessão roda até o teto de parede e para. É UM notebook, que detecta o
hospedeiro na célula do repo (`COLAB = /content existe e /kaggle não`).

⚠⚠ **Este notebook NÃO continua nada.** Quem continua uma linhagem é o
`g1_limpo_kaggle.ipynb`, que procura o `model_*.pt` de maior número num dataset de
entrada. Rodar ESTE de novo recomeça com pesos aleatórios e joga fora a sessão
anterior — a continuação é o outro arquivo.

## Antes de rodar — na Kaggle

1. **Settings → Accelerator → GPU** (T4 ×1 ou P100).
2. **Settings → Internet → On.** O clone do GitHub e a subida do dataset precisam dela.
3. **NÃO anexe dataset de entrada.** Do zero não há o que ler; nenhuma célula abaixo
   olha para `/kaggle/input`.
4. **A branch tem de estar no GitHub.** Na sua máquina:
   `git push -u origin exp/g1-limpo-v2`

## Antes de rodar — no Colab

1. Abra este arquivo direto do GitHub:
   `https://colab.research.google.com/github/JoaoBornelli/g1_training/blob/exp/g1-limpo-v2/g1_limpo/kaggle/g1_limpo_zero.ipynb`
2. **Runtime → Change runtime type → GPU.**
3. Nada a subir no Drive antes: ele é só SAÍDA aqui.
4. A branch tem de estar no GitHub, igual à Kaggle.

## O que muda por hospedeiro

| | Kaggle | Colab |
|---|---|---|
| clone / log / zip | `/kaggle/working/…` | `/content/…` — **local, não no Drive** |
| checkpoint de saída | `kaggle datasets version` (célula da API) | cópia para `MyDrive/g1_limpo`: no `finally` e a cada 10 min |
| segredos | `KAGGLE_USERNAME` / `KAGGLE_KEY` | nenhum — o Drive é seu |
| GPU | 16 GB (T4: 19 307 passos/s a 4096) | 16 GB (23 022 passos/s a 8192, bloco8) |
| `NUM_ENVS` | 4096 | **8192** |
| sessão | **12 h duras** | gratuito cai por ociosidade; Pro vai a 24 h |

**`NUM_ENVS` é pelo hospedeiro, não pela VRAM.** As duas GPUs têm 16 GB; a do Colab roda
8192 envs e a da Kaggle não — medido pelo dono nas duas. Não há limiar de memória a
inventar.

## As três diferenças em relação ao `g1_limpo_kaggle.ipynb`

- **As iterações vêm SÓ do relógio.** Não há iteração de destino a alcançar: a sessão
  roda o que couber em `HORAS_LIMITE` e para. A 4096 envs na Kaggle, ~5 478 iterações.
- **A tabela do `limite_de_junta` é a DO ZERO**, e esta é a mudança central. A rampa
  acaba NO BATENTE nas três famílias (`k = 20`, teto `0,15`), e não em 155% do curso
  como a tabela larga que o knob traz por padrão. A tabela larga existe para consertar
  política viciada e aceita o batente como preço; uma política que nasce do zero tem de
  aprender a não chegar nele. O override mora na célula do treino, não no knob.
- **A saída vai para o dataset `g1-limpo-zero`**, e nunca para o `g1-limpo-v2`. O
  motivo é medido e está no comentário `⚠⚠` da célula da API.

## O log nasce vazio

Não há semeadura de árvore de log, e é isso que um treino do zero quer: o runner do
mjlab cria a pasta de run sozinho, com o timestamp de agora.

## Para persistir

Na Kaggle, rode com **Save Version → Save & Run All (Commit)**, ou use a célula da API no
fim. Sessão interativa que expira leva `/kaggle/working` embora — foi assim que o bloco 4
se perdeu, e uma queda de internet levou 1500 iterações.

No Colab, a cópia periódica para `MyDrive/g1_limpo/em_curso/` limita a perda a 10 min.

In [ ]:
# ⚠ NADA DE `import torch` NESTA CÉLULA. O torch registra operadores C++ no import,
# e se ele entrar no kernel ANTES do pip, um reload depois levanta
# `Only a single TORCH_LIBRARY can be used to register the namespace triton`.
import subprocess, sys

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "sem GPU. Kaggle: Settings -> Accelerator -> GPU. " \
    "Colab: Runtime -> Change runtime type -> GPU"
print("python", sys.version.split()[0])

## Dependências

**Instale só o `mjlab`.** Ele declara a árvore inteira e já pina o que importa:
`mujoco-warp~=3.10.0,>=3.10.0.3`, `mujoco~=3.10.0`, `rsl-rl-lib==5.4.0`, `numpy<2.5`,
`tensorboard>=2.20.0`, `scipy>=1.15`. Repetir esses pins à mão não adiciona segurança —
adiciona modos de falha, porque uma única versão indisponível derruba a resolução toda.

⚠ **A versão é 1.5.3, não 1.5.1.** O `RslRlModelCfg` de 1.5.1 manda `cnn_cfg` e
`rnn_type` que o `MLPModel` do `rsl-rl-lib 5.4.0` rejeita. O
`g1_multitask/kaggle/requirements.txt` ainda pede 1.5.1 — ele está velho, não o copie.

⚠ **`torch` não entra.** A imagem da Kaggle traz um build casado com o CUDA da máquina;
deixar o pip trocá-lo é o jeito mais rápido de perder a GPU sem perceber. O `mjlab`
exige `torch>=2.7.0` e as imagens atuais satisfazem.

In [ ]:
import subprocess, sys

# ⚠⚠ LISTA DE ARGUMENTOS, NUNCA STRING DE SHELL. Este foi um defeito medido: com
# `!pip install ... scipy>=1.15 numpy<2.5` o shell lê `>` e `<` como REDIRECIONAMENTO,
# tenta abrir um arquivo chamado `2.5`, aborta com exit 2 — e o pip NUNCA RODA. Com
# `-q` e sem conferir o código de saída, a célula não reclama e o erro só aparece duas
# células depois, como `No module named 'mjlab'`.
cmd = [sys.executable, "-m", "pip", "install", "--no-warn-conflicts", "mjlab==1.5.3"]
print(" ".join(cmd), flush=True)
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
assert r.returncode == 0, (
    "o pip falhou. A causa mais comum é a internet DESLIGADA: "
    "Settings -> Internet -> On, e rode esta célula de novo.")

r = subprocess.run([sys.executable, "-m", "pip", "list"],
                   capture_output=True, text=True)
alvo = ("mjlab", "rsl-rl-lib", "mujoco", "mujoco-warp", "warp-lang", "torch",
        "tensordict", "tyro", "tensorboard", "numpy", "scipy")
for linha in r.stdout.splitlines():
    if linha.split(" ")[0].lower() in alvo:
        print(linha)

## O pip levou a GPU?

Se ele trocou o torch por um build sem CUDA, **tudo abaixo roda em CPU sem reclamar** e
a sessão vira 12 h de nada. A checagem roda num subprocesso, porque `reload(torch)` não
existe.

In [ ]:
import subprocess, sys

chk = subprocess.run(
    [sys.executable, "-c",
     "import torch;print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)"],
    capture_output=True, text=True)
print("subprocesso:", chk.stdout.strip() or chk.stderr[-600:])
assert " True " in f" {chk.stdout} ", (
    "o pip trocou o torch e a CUDA foi embora. Reinstale com --no-deps o que puxou "
    "torch, ou reinicie o kernel e rode da célula 1")

# só agora, e é a PRIMEIRA vez neste processo
import torch, mjlab, mujoco, warp
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"{torch.cuda.get_device_name(0)}")
print("mjlab", mjlab.__version__ if hasattr(mjlab, "__version__") else "?",
      "| mujoco", mujoco.__version__, "| warp", warp.config.version)

## O repo, e o log vazio

A célula abaixo clona a branch, põe o clone no `sys.path` e abre o `LOG_ROOT` — vazio,
que é o estado certo para um treino do zero.

⚠ **Se você deu `git push` depois de abrir esta sessão, RE-RODE esta célula.** A Kaggle
já rodou código antigo sem avisar uma vez; o sinal foi um peso de knob velho no primeiro
print.

⚠ **A task é registrada por efeito colateral do `import g1_limpo`.** Editar o arquivo
não basta: o kernel usa a versão em cache. Num commit isso não acontece (kernel novo);
numa sessão interativa, reinicie o runtime e rode da célula 1.

In [ ]:
import importlib, os, pathlib, shutil, subprocess, sys

os.environ.setdefault("MUJOCO_GL", "egl")

# ⚠ O HOSPEDEIRO, ANTES DE QUALQUER CAMINHO. Não é `import google.colab`: fora do Colab
# esse import falha com uma mensagem que confunde. `/content` existe no Colab, e
# `/kaggle` só na Kaggle.
COLAB = pathlib.Path("/content").is_dir() and not pathlib.Path("/kaggle").is_dir()
print("hospedeiro =", "Colab" if COLAB else "Kaggle")

# ⚠ 16/09: entrou o `forma_postural` (incentivo de forma contra a referência da IK,
# `g1_limpo/ik/ref_botar.npz`), gateado nas idas da pega e do pouso. Do zero ele age
# desde a primeira iteração, junto com a rampa 0,85–1,00 do `limite_de_junta`.
RUN      = "zero01"             # ⚠ NOME NOVO A CADA MUDANÇA DE CONJUNTO, e não a cada
                                # mudança de peso. O `RUN` é a impressão digital da
                                # tabela de recompensa, e ele mantém duas tabelas
                                # diferentes fora do MESMO log, num gráfico só.
                                #
                                # ⚠⚠ `zero01` É UMA LINHAGEM NOVA, e não a continuação
                                # dos `blocoNN`. Todos eles descendem do mesmo
                                # `model_*.pt`; este nasce com pesos aleatórios, e os
                                # dois não se comparam iteração a iteração.
                                #
                                # ⚠ O dono autorizou SOBRESCREVER a `zero01` antiga
                                # (2026-09-15): ela quase não rodou e não é referência.
# ⚠⚠ A MESMA BRANCH do notebook de continuação. Ela nasce da tag `estavel-bloco17` e
# traz o `limite_de_junta` no lugar do `dof_pos_limits` do fabricante — é o termo cuja
# TABELA este notebook troca, na célula do treino. O código é o mesmo nos dois; o que
# muda é o valor que o notebook escreve no `cfg`.
BRANCH   = "exp/g1-limpo-v2"                 # ⚠ 16/09: a linha do worktree virou a oficial

# ⚠ OS CAMINHOS SÃO LOCAIS NOS DOIS. No Colab o log NÃO vai para o Drive: o `tfevents` é
# anexado a cada iteração, e o FUSE do Drive reescreve o arquivo inteiro a cada flush —
# lento, e já viu corromper. O Drive recebe CÓPIAS (célula do pacote e do treino).
BASE     = pathlib.Path("/content" if COLAB else "/kaggle/working")
RAIZ     = BASE / "g1"                            # o clone, refeito a cada sessão
LOG_ROOT = BASE / "logs"                          # ⚠ FORA de RAIZ
raiz_exp = LOG_ROOT / "g1_limpo"                  # <log_root>/<experiment_name>

# ------------------------------------------------------------------- 1. o clone
# ⚠ EXIGE INTERNET LIGADA e a branch NO GITHUB.
if RAIZ.exists():
    shutil.rmtree(RAIZ)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",
                "https://github.com/JoaoBornelli/g1_training.git", str(RAIZ)],
               check=True)
print("clone   =", subprocess.run(["git", "-C", str(RAIZ), "log", "--oneline", "-1"],
                                  capture_output=True, text=True).stdout.strip())

# ⚠ `invalidate_caches` NÃO é higiene. O Python guarda um finder POR DIRETÓRIO em
# `sys.path_importer_cache`, e o finder de um diretório que não existia no momento da
# inserção fica cacheado como VAZIO — `import g1_limpo` falharia com `No module named`
# mesmo com o pacote em disco.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
importlib.invalidate_caches()

# --------------------------------------------- 2. o Drive, e SÓ para a SAÍDA
# ⚠ NÃO HÁ PASTA DE ORIGEM NESTE NOTEBOOK. O treino do zero não lê `model_*.pt` de
# lugar nenhum: nem de `/kaggle/input`, nem do Drive. A montagem existe porque o
# `empacota()` e a cópia periódica ESCREVEM lá.
if COLAB:
    # ⚠ Uma linha, só aqui. Nenhum segredo: o Drive é do próprio usuário. A montagem
    # pede autorização interativa na primeira vez.
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = pathlib.Path("/content/drive/MyDrive/g1_limpo"); DRIVE.mkdir(exist_ok=True)
else:
    DRIVE = None

# ----------------------------------- 3. o log NASCE VAZIO, e é isso que se quer
# ⚠⚠ NADA DE SEMENTE `1900-01-01`. Aquela pasta existe no notebook de continuação para
# dar ao mjlab a árvore `<log_root>/g1_limpo/<run_dir>/<ckpt>` que ele sabe procurar,
# a partir de uma pasta de origem PLANA (dataset da Kaggle, pasta do Drive). Do zero não há checkpoint a semear: o runner do
# mjlab cria a pasta de run sozinho, com o timestamp de agora.
LOG_ROOT.mkdir(parents=True, exist_ok=True)

# ⚠ AVISO, e não assert: a célula acima manda RE-RODAR esta depois de um `git push`, e
# um assert aqui travaria justamente isso. Mas uma pasta de run desta `RUN` já em disco
# significa que a sessão JÁ TREINOU — e lançar o treino de novo escreve uma segunda
# pasta com pesos aleatórios, ao lado da primeira.
ja_rodou = sorted(p for p in raiz_exp.glob(f"*_{RUN}") if p.is_dir())
if ja_rodou:
    print(f"\n⚠⚠ JÁ EXISTE run de {RUN!r} neste log: {[p.name for p in ja_rodou]}")
    print("   Rodar a célula do treino de novo COMEÇA OUTRA VEZ do zero, numa pasta")
    print("   nova, e não continua esta. Suba o checkpoint (célula da API) antes.")

achados = sorted(raiz_exp.rglob("*"))
print(f"\nlog_root = {LOG_ROOT}")
for p in achados:
    print("  ", p.relative_to(LOG_ROOT))
if not achados:
    print("   (vazio — é o esperado num treino do zero)")

## O pacote de saída, e o download

⚠ **Você não está commitando a versão, portanto `/kaggle/working` MORRE com a sessão.**
Esta célula define `empacota()`, e a célula do treino a chama num `finally` — o pacote
nasce no fim normal do treino E se ele estourar ou se você interromper o kernel.

⚠⚠ **O que código nenhum salva é a sessão MORTA** (12 h, timeout de inatividade, aba
fechada). Aí nada roda, nem isto. Para não depender disso, veja a célula do fim: subir
para um Dataset pela API da Kaggle é a única persistência que funciona sem você estar
presente.

In [ ]:
# =====================================================================
#  EMPACOTA E BAIXA — rode DEPOIS do treino, ou chame no `finally` dele.
#  E, no Colab, as CÓPIAS para o Drive: a final e a periódica.
# =====================================================================
import os, pathlib, re, shutil, threading, time, zipfile
from IPython.display import FileLink, HTML, display

COLAB    = pathlib.Path("/content").is_dir() and not pathlib.Path("/kaggle").is_dir()
RUN      = "bloco20"
BASE     = pathlib.Path("/content" if COLAB else "/kaggle/working")
LOG_ROOT = BASE / "logs"
SAIDA    = BASE                                   # onde o zip nasce
DRIVE    = pathlib.Path("/content/drive/MyDrive/g1_limpo") if COLAB else None
ULTIMO_CKPT = ULTIMA_RUN = ULTIMA_IT = ULTIMO_PACOTE = None


def _run_nova(run=RUN):
    """A pasta de run REAL mais recente e seus `model_*.pt` por número — ou `(None, [])`.

    ⚠ Nunca uma pasta `1900-...`. ESTE notebook não semeia nenhuma — quem semeia é o
    de continuação, e o filtro fica aqui para as duas buscas serem A MESMA. UMA busca,
    para o `empacota`, a cópia periódica e a célula do fim não divergirem entre si.
    """
    raiz_exp = LOG_ROOT / "g1_limpo"
    if not raiz_exp.is_dir():
        return None, []
    runs = sorted(p for p in raiz_exp.iterdir()
                  if p.is_dir() and p.name.endswith(run)
                  and not p.name.startswith("1900"))
    if not runs:
        return None, []
    cks = sorted(runs[-1].glob("model_*.pt"),
                 key=lambda p: int(re.search(r"(\d+)", p.name).group(1)))
    return runs[-1], cks


def empacota(run=RUN, baixa=True):
    """Zipa o último checkpoint + tfevents + params e oferece o download.

    ⚠ UM ZIP SÓ, e não os arquivos soltos. Um bloco de 1500 iterações deixa ~30
    checkpoints; baixar um por um é o que faz a sessão expirar no meio. E o tfevents
    vai junto porque sem ele o `leitura.py` não tem o que ler.
    """
    raiz_exp = LOG_ROOT / "g1_limpo"
    if not raiz_exp.is_dir():
        print(f"nada em {raiz_exp} — o treino não chegou a escrever")
        return None

    nova, cks = _run_nova(run)
    if nova is None:
        print(f"nenhuma pasta de run nova em {raiz_exp} — o treino não salvou nada")
        return None
    if not cks:
        print(f"nenhum checkpoint em {nova.name} — nem 50 iterações rodaram")
        return None
    ultimo = cks[-1]
    it = int(re.search(r"(\d+)", ultimo.name).group(1))

    pacote = SAIDA / f"{run}_it{it}.zip"
    with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ultimo, ultimo.name)
        for ev in sorted(nova.glob("events.out.tfevents*")):
            z.write(ev, ev.name)
        dig = raiz_exp / f"{run}.pesos.json"
        if dig.exists():
            z.write(dig, dig.name)
        for p in sorted((nova / "params").glob("*.yaml")):
            z.write(p, f"params/{p.name}")
        # a lista do que FICOU para trás, para você saber o que perdeu se a sessão morrer
        z.writestr("INVENTARIO.txt",
                   f"run: {nova.name}\nultimo: {ultimo.name}\n"
                   f"checkpoints na sessao: {len(cks)}\n"
                   + "\n".join(p.name for p in cks) + "\n")

    print(f"run       = {nova.name}")
    print(f"salvos    = {len(cks)} checkpoints ({cks[0].name} .. {ultimo.name})")
    print(f"pacote    = {pacote.name}  ({pacote.stat().st_size / 2**20:.1f} MB)")
    print(f"conteudo  = {zipfile.ZipFile(pacote).namelist()}")
    # ⚠ GLOBAIS, para as células de upload e de Drive não repetirem a busca (e não
    # divergirem dela).
    global ULTIMO_CKPT, ULTIMA_RUN, ULTIMA_IT, ULTIMO_PACOTE
    ULTIMO_CKPT, ULTIMA_RUN, ULTIMA_IT, ULTIMO_PACOTE = ultimo, nova, it, pacote
    print(f"\nna proxima sessao: a célula do repo acha {ultimo.name!r} pelo maior número")

    if baixa:
        # ⚠ O href TEM de ser relativo. O `FileLink` monta o link a partir do cwd do
        # kernel (`/kaggle/working` na Kaggle, `/content` no Colab); caminho absoluto
        # gera href quebrado que abre uma pagina de erro. O `chdir` garante o cwd.
        os.chdir(SAIDA)
        display(FileLink(pacote.name))
        # ⚠ TENTATIVA de clique automatico. O output roda em iframe com sandbox, e
        # download iniciado por script pode ser BLOQUEADO sem mensagem.
        # O link acima e o caminho garantido — nao confie so nisto.
        display(HTML(
            f'<a id="bx{it}" href="{pacote.name}" download="{pacote.name}"></a>'
            f'<script>document.getElementById("bx{it}").click();</script>'
            f'<p><b>Se nada baixou</b>: clique no link acima, ou Data &rarr; Output '
            f'&rarr; <code>{pacote.name}</code> (Colab: painel Files, ou o Drive).</p>'))
    return pacote


def copia_para_drive():
    """Colab: o último `model_*.pt`, o `.pesos.json` e o zip vão para `MyDrive/g1_limpo`.

    ⚠ CÓPIAS, nunca o `LOG_ROOT`: o FUSE do Drive reescreve o arquivo inteiro a cada
    flush do `tfevents`. É a versão FINAL, e é o que a célula do repo acha na próxima
    sessão. Redundante com a `em_curso/` da cópia periódica, de propósito.
    """
    assert COLAB, "não é Colab: na Kaggle a persistência é a célula da API do dataset"
    assert DRIVE.parent.is_dir(), "Drive não montado — rode a célula do repo"
    if ULTIMO_CKPT is None:
        print("nada para copiar: o `empacota()` não achou checkpoint")
        return
    DRIVE.mkdir(exist_ok=True)
    for p in (ULTIMO_CKPT, LOG_ROOT / "g1_limpo" / f"{RUN}.pesos.json", ULTIMO_PACOTE):
        if p is not None and p.exists():
            shutil.copy2(p, DRIVE / p.name)
            print(f"drive     = {DRIVE / p.name}")
    print(f"\nna proxima sessao: a célula do repo acha {ULTIMO_CKPT.name!r} em MyDrive/g1_limpo")


def copia_periodica_para_drive(intervalo_s=600):
    """Colab: thread daemon que, a cada 10 min, copia para `DRIVE/em_curso/` o último
    `model_*.pt` da run nova — se ele mudou.

    ⚠ Contra a QUEDA da sessão: a Kaggle caiu por internet e levou `/kaggle/working`
    inteiro, 1500 iterações. Com isto a perda máxima passa a ser 10 min.
    ⚠ Só o ÚLTIMO checkpoint, nunca a pasta: 30 checkpoints de 6 MB a cada 50 iterações
    seriam 30 uploads. `shutil.copy2`, e a comparação é pelo `st_mtime` (nome + mtime do
    último copiado): mesmo nome e mesmo mtime, nada a fazer.
    ⚠ Daemon: morre com o kernel e não segura o `finally` do treino.
    """
    assert COLAB, "a cópia periódica é só do Colab; na Kaggle é a API do dataset"

    def _laco():
        visto = None                       # (nome, st_mtime) do último copiado
        while True:
            time.sleep(intervalo_s)
            try:
                _, cks = _run_nova()
                if not cks:
                    continue
                ultimo = cks[-1]
                chave = (ultimo.name, ultimo.stat().st_mtime)
                if chave == visto:
                    continue
                # ⚠ `torch.save` não é atômico: um .pt escrito há menos de 30 s pode
                # estar pela metade. Fica para o próximo tique.
                if time.time() - chave[1] < 30:
                    continue
                destino = DRIVE / "em_curso" / ultimo.name
                destino.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(ultimo, destino)
                visto = chave
                print(f"[drive] em_curso/{ultimo.name}", flush=True)
            except Exception as e:      # a thread NUNCA derruba o treino
                print(f"[drive] cópia falhou: {e!r}", flush=True)

    t = threading.Thread(target=_laco, daemon=True, name="copia_drive")
    t.start()
    return t


# roda de graça se ainda não há nada: só imprime e sai
empacota()

## O treino do zero

Idêntico ao da continuação nos asserts de paridade do clone, com três diferenças: a
tabela do `limite_de_junta` vai para o valor do zero, `max_iterations` vem só do
relógio, e a LR fica no default de 1e-3 — a queda para 5e-4 é regra de warm-start, e
aqui não há política velha para desmanchar.

**Sobre a segunda T4.** Não está ligada, e o motivo é ARITMÉTICO, não medo. O tempo de
iteração é 89% física e 10% update, e os DOIS são data-parallel: com 4096 envs POR RANK
você ganha o LOTE de volta (196 608 transições, o da bloco8) com o mesmo tempo de
iteração — e **zero** iterações por hora a mais. Para ganhar iterações/hora seria preciso
2048 por rank, e aí cada T4 fica sub-utilizada e o ganho cai para ~1,5×.

Somando: um T4 com 4096 envs a ~6,9 s/iter fecha ~5 478 iterações em 10,5 h, dentro de
uma sessão. A segunda GPU compra lote, não tempo. E cobra: o estado do currículo
(`env.limpo_forma`) é POR PROCESSO, portanto os dois ranks tocam rampas independentes e
só o rank 0 vai para o checkpoint; o seed do rank 1 é 43 e não 42; e o log dos dois
ranks inunda a célula. Se quiser mesmo, `cfg.gpu_ids = "all"` e `NUM_ENVS` é **por
rank**.

In [ ]:
# =====================================================================
#  TREINO DO ZERO — g1_limpo. O run vem de `RUN`, acima.  KAGGLE OU COLAB.
# =====================================================================
import dataclasses, json, sys
sys.path.insert(0, str(RAIZ))

import g1_limpo
from g1_limpo import comando as CMD, knobs as KN, observacoes as OB
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.tasks.velocity.config.g1.env_cfgs import unitree_g1_flat_env_cfg

# ⚠ `NUM_ENVS` É PELO HOSPEDEIRO, E NÃO PELA VRAM. Medido pelo dono nas duas
# plataformas (2026-09-11): a GPU do Colab tem 16 GB e RODA 8192 envs; a da Kaggle,
# também de 16 GB, NÃO. A VRAM nominal não explica, portanto o discriminador é o
# hospedeiro, e não um número de `nvidia-smi`. Não invente um limiar de memória.
# Com 4096 o LOTE do PPO cai pela metade (`num_envs × num_steps_per_env`) e, com
# `num_mini_batches = 4`, o minilote vai de 49 152 para 24 576 transições. Com 8192
# ele volta a 196 608 / 49 152 — o lote da bloco8. Se der OOM na Kaggle: 2048.
NUM_ENVS   = 8192 if COLAB else 4096

# ⚠⚠ O TETO DE PAREDE, E NUM TREINO DO ZERO ELE É A ÚNICA TRAVA. Não existe iteração
# de destino aqui: não há de onde contar. A sessão roda o que couber no relógio e para,
# e a continuação é outro notebook. Na Kaggle o teto é DURO: 12 h, e sem commit a
# sessão morta leva `/kaggle/working`. No Colab o gratuito cai por OCIOSIDADE antes das
# 10,5 h e o Pro vai a 24 h — o dono ajusta `HORAS_LIMITE` aqui.
#
# O `SEG_POR_ITER` está ANCORADO NUMA MEDIÇÃO, e não num chute:
#
#   4096 envs, 1× T4, g1_limpo, log do bloco19: 14 265 passos/s com 24 passos por env,
#       ou seja 4096 × 24 = 98 304 passos por iteração -> 98304/14265 = 6,89 s/iter.
#       É a medição DESTA cena (mesa, caixa, 29 termos), e não a do multitask.
#   8192 envs, a bloco8 no Colab, GPU de 16 GB: 7,554 + 0,986 = 8,54 s/iter, 23 022
#       passos/s. A bloco8 RODOU NO COLAB (dono, 2026-09-11); os 23 022 são só 1,19×
#       os 19 307 de um T4 com 4096 — não era A100. O 9,0 abaixo é a margem.
#
# ⚠ O 7,0 DA KAGGLE DESCEU PARA 6,9 (medição do bloco19 acima). A 10,5 h isso dá
# ~5 478 iterações por sessão, e não ~5 400.
# ⚠ Nos dois hospedeiros, CORRIJA com o `Collection time` real do primeiro log.
HORAS_LIMITE = 10.5   # Kaggle: 12 h menos o pip, o clone e a montagem do env
SEG_POR_ITER = 9.0 if COLAB else 6.9

cfg = dataclasses.replace(TrainConfig.from_task(g1_limpo.TASK_ID),
                          log_root=str(LOG_ROOT))
cfg.env.scene.num_envs = NUM_ENVS
cfg.agent.run_name = RUN
cfg.agent.logger = "tensorboard"

# ------------------------------------------------- AS ITERAÇÕES VÊM DO RELÓGIO
# ⚠ `max_iterations` É ADITIVO no rsl_rl (`on_policy_runner.py:78` faz
#     total_it = current_learning_iteration + num_learning_iterations),
# mas do zero o `current_learning_iteration` é 0 — portanto aqui ele é o número final.
cabe_tempo = int(HORAS_LIMITE * 3600 / SEG_POR_ITER)
cfg.agent.max_iterations = cabe_tempo
print(f"cabe em {HORAS_LIMITE:.1f} h a {SEG_POR_ITER:.1f} s/iter = {cabe_tempo}")
print(f"vai rodar   = {cfg.agent.max_iterations} iterações (manda o RELÓGIO) "
      f"-> termina na {cfg.agent.max_iterations}")
print("\nmarcos esperados, para saber o que olhar:")
print("  ~400   fica de pé")
print("  ~1000  a marcha se forma (razao_marcha acima de 0,50)")
print("  ~2500  a fatia de locomoção começa a descer de 0,95")

# ------------------------------------------------------------------ A LR
# ⚠⚠ A LR FICA EM 1e-3, e é aqui que este notebook se separa do de continuação. Lá ela
# desce para 5e-4 porque a tabela de recompensa mudou debaixo de uma política treinada:
# é WARM-START, a função de valor está errada por vários pontos por segundo, e a nota
# `warmstart-lower-lr` manda baixar. Do zero não há política nem função de valor velha
# — a regra não se aplica, e baixar a LR só faria a rampa dos 400 primeiros custar mais.
assert cfg.agent.algorithm.schedule == "adaptive"
assert cfg.agent.algorithm.learning_rate == 1.0e-3, (
    f"a LR do molde não é 1e-3 ({cfg.agent.algorithm.learning_rate}): alguém já mexeu "
    "nela, e do zero ela tem de ser a do fabricante")
assert cfg.agent.seed == 42, \
    "o seed é 42 em toda a linhagem; trocá-lo torna esta run incomparável com as outras"

# =====================================================================
#  A TABELA DO `limite_de_junta` VAI PARA O VALOR DO ZERO
# =====================================================================
# ⚠⚠ ESTA É A MUDANÇA CENTRAL DO NOTEBOOK. O knob `LimiteDeJunta` traz por padrão a
# tabela LARGA — `tornozelo (4.5, 0.70)`, `punho_cintura (12.0, 0.25)` —, cuja rampa
# acaba em 155% e em 110% do meio-curso. Ela existe para CONSERTAR uma política que JÁ
# APRENDEU a usar curso que não existe, e o próprio docstring do knob declara o preço:
# no tornozelo o BATENTE passa a custar 0,96, e não 19. O termo diz "volte para dentro
# do curso", e não "não encoste no batente".
#
# Uma política que nasce do zero não tem esse vício, e tem de aprender a NÃO CHEGAR ao
# batente. Para isso a rampa acaba NELE, nas três famílias — é o que o docstring manda
# em letras maiúsculas: `tornozelo = punho_cintura = resto = (20.0, 0.15)`, rampa de
# 0,85 a 1,00, com `k = 20` e teto 0,15.
#
# ⚠ POR QUE O OVERRIDE MORA AQUI E NÃO NO KNOB. A task é registrada na IMPORTAÇÃO
# (`g1_limpo/__init__.py`, `register_mjlab_task(env_cfg=make_env_cfg())`), com os knobs
# padrão, e `TrainConfig.from_task` não aceita knobs. Mutar o `cfg` depois é o idioma
# deste notebook, e é SEGURO: `load_env_cfg` devolve `deepcopy` do registro
# (`mjlab/tasks/registry.py:53`), portanto isto não contamina o registro, nem as tasks
# de inspeção, nem o notebook de continuação — que continua precisando da tabela larga.
#
# ⚠ A `LimiteDeJunta` de `recompensas.py` resolve a tabela no `__init__`, quando o
# reward manager instancia o termo — ou seja, DEPOIS desta linha e antes do primeiro
# passo. Trocar o dict aqui basta.
_lj = dataclasses.replace(KN.LimiteDeJunta(),
                          tornozelo=(20.0, 0.15),
                          punho_cintura=(20.0, 0.15),
                          resto=(20.0, 0.15))
cfg.env.rewards["limite_de_junta"].params["tabela"] = _lj.por_padrao()

# ⚠⚠ O ASSERT QUE PROVA A TROCA. Sem ele o modo de falha é SILENCIOSO: o treino do zero
# rodaria com o freio largo, e o próprio termo ensinaria a política a encostar no
# batente — que é exatamente o que se quer que ela nunca aprenda.
_tab = cfg.env.rewards["limite_de_junta"].params["tabela"]
assert len(_tab) == 14, \
    f"`por_padrao` abre as 14 famílias de `FAMILIAS`, e a tabela tem {len(_tab)}: {_tab}"
_largas = {p: v for p, v in _tab.items()
           if abs(v[0] - 20.0) > 1e-9 or abs(v[1] - 0.15) > 1e-9}
assert not _largas, (
    "a tabela do `limite_de_junta` NÃO é a do zero. As 14 famílias têm de ler "
    f"k = 20,0 e teto = 0,15, e estas não leem: {_largas}. A tabela LARGA "
    "(tornozelo 4,5/0,70, punho_cintura 12,0/0,25) acaba em 155% e em 110% do "
    "meio-curso: ela conserta política viciada e ACEITA o batente como preço. Do "
    "zero a rampa tem de acabar NO batente.")
_limiar = cfg.env.rewards["limite_de_junta"].params["limiar"]
assert _limiar == 0.85 == _lj.limiar, (
    f"o limiar da rampa é {_limiar}, e tem de ser 0,85. Ele é o MESMO nos dois "
    "notebooks — do zero o que muda é só onde a rampa TERMINA, não onde ela começa.")
print(f"\nlimite_de_junta = rampa {_limiar:.2f} a 1,00 nas {len(_tab)} famílias "
      f"(k 20,0, teto 0,15) — a tabela DO ZERO")

# =====================================================================
#  DAQUI PARA BAIXO É IDÊNTICO À CÉLULA DE TREINO DA CONTINUAÇÃO. O clone é
#  NOVO em toda sessão, portanto os asserts de "é a v2 mesmo?" valem igual.
# =====================================================================
rw, cu, tm, ev = (cfg.env.rewards, cfg.env.curriculum,
                  cfg.env.terminations, cfg.env.events)
fab = unitree_g1_flat_env_cfg(play=False)

# --- O CLONE É A v2? Estes cinco falham alto num clone antigo. ---
assert CMD.DIM == 12 and CMD.GIRO == slice(9, 12), \
    f"comando com DIM={CMD.DIM}: clone anterior ao canal `giro_b` (spec §8.3)"
assert OB.N_CAIXA == 10, f"N_CAIXA={OB.N_CAIXA}: clone sem `giro_b`/`meia_aresta`"
assert len(CMD.CADEIAS) == 3 and all(CMD.CARREGAR not in c for c in CMD.CADEIAS), \
    f"CADEIAS = {CMD.CADEIAS}: clone anterior ao dois-bits (CARREGAR virou CAUDA, spec §2.1)"
# ⚠ v2.1: o `load` SAIU e o `renda_congelada` entrou. O congelamento paga TODO fecho,
# portanto o fecho do BOTAR rende mais que pairar sem o `load` — e sem número escolhido
# à mão. Ver docs/planos/2026-09-04-proposta-gradientes-g1-limpo.md §3 P3.
assert "largou" not in rw and "load" in rw, \
    f"clone fora do dois-bits: load={'load' in rw}, largou={'largou' in rw}"
assert list(rw)[-1] == "renda_congelada", \
    f"`renda_congelada` tem de ser o ÚLTIMO termo (ele lê `_step_reward` dos outros); hoje o último é {list(rw)[-1]!r}"
assert "tamanho_caixa" in ev and ev["tamanho_caixa"].mode == "startup", \
    "sem o evento de startup a caixa tem tamanho fixo e a obs lê zero no `meia_aresta`"

# --- contrato do pacote ---
assert list(cfg.env.commands) == ["twist", "alvo_caixa"], list(cfg.env.commands)
assert cfg.env.commands["alvo_caixa"].resampling_time_range[0] > cfg.env.episode_length_s, \
    "resample dentro do episódio zera o sucesso um passo antes do time_out"
assert tm["time_out"].time_out is True, "o rsl_rl trataria time_out como fracasso"
assert list(cu) == ["command_vel", "forma", "nivel", "elo"], \
    f"ordem do currículo errada ({list(cu)}): `forma` e `nivel` leriam o elo do episódio SEGUINTE"
assert cfg.agent.experiment_name == g1_limpo.EXPERIMENT == "g1_limpo"

# --- a OBSERVAÇÃO da v2: ator 114, crítico 131 (114 + 12 de pé do fabricante + 5) ---
assert list(cfg.env.observations["actor"].terms)[-2:] == ["elo", "caixa"], \
    list(cfg.env.observations["actor"].terms)
assert list(cfg.env.observations["critic"].terms)[-3:] == ["elo", "caixa", "elo_interno"], \
    "sem `elo_interno` o crítico confunde a espera final com um env parado, e o " \
    "`PPOPorElo` agrupa a espera final na locomoção (spec §6.1)"

# --- PARIDADE: a locomoção é a do fabricante ---
# ⚠ 17 termos nossos: os 8 da tarefa, as 3 multas de mesa (viraram multa em 01/09, e
# ANTES eram terminação), o `renda_congelada` da v2.1, o `velocidade_por_regime` da G2,
# a `faixa_de_pose` da G3, o `limite_de_junta` do bloco 18 e o `limite_de_pelve` do
# bloco 20 (preço quadrático na altura da pelve, gateado no CARREGAR — plano
# `docs/planos/2026-09-15-limite-de-pelve-no-carregar.md`).
assert set(rw) - set(fab.rewards) == {
    "staged", "precise_pos", "precise_ori", "squeeze", "unload",
    "postura_ereta", "load", "terminacao", "joint_acc",
    "contato_tronco", "contato_palma", "contato_dorso",
    "renda_congelada", "velocidade_por_regime", "faixa_de_pose",
    "limite_de_junta", "limite_de_pelve", "forma_postural"}, \
    f"termo inventado na locomoção: {sorted(set(rw) - set(fab.rewards))}"
# ⚠⚠ UM TERMO DO MOLDE SAI, e é o único até hoje. O `dof_pos_limits` cobrava o excesso
# LINEAR acima de 90% do curso, e o `limite_de_junta` cobra o MESMO excesso com rampa
# exponencial — os dois juntos seriam cobrança dupla. O nome está DECLARADO aqui de
# propósito: um `not set(...)` solto deixaria de pegar o dia em que um upgrade do mjlab
# apagar outro termo em silêncio.
assert set(fab.rewards) - set(rw) == {"dof_pos_limits"}, \
    f"termo do fabricante desapareceu: {sorted(set(fab.rewards) - set(rw))}"
assert cfg.env.actions["joint_pos"].scale == fab.actions["joint_pos"].scale
# ⚠ PARIDADE QUEBRADA DE PROPÓSITO, em DUAS frentes.
# 1) O TERCEIRO ESTÁGIO SAI (spec `g1-limpo-giro-e-teclas.md` §0b): o molde sobe
#    `lin_vel_x` a 3,0 m/s na iteração 10000, por passo fixo. MEDIDO na bloco14:
#    `s_C` 0,195 -> 0,142 e `caixa_largada` 10,7% -> 16,3% ao cruzar.
# 2) O ENVELOPE DE GUINADA SOBE (spec `g1-limpo-envelope-de-giro.md`): o dono pediu
#    uma volta completa em no máximo 4 s. `2π/4 = 1,5708 rad/s`, e o teto é ±1,6
#    (volta em 3,93 s). O molde para em ±0,7 — 8,98 s por volta. O estágio 0 fica
#    INTOCADO em ±0,5: ele é a rampa de uma run do ZERO, e esta É uma run do zero —
#    aqui esse estágio não é teoria, é o envelope do primeiro passo.
_ests = cu["command_vel"].params["velocity_stages"]
_fab_ests = [e for e in fab.curriculum["command_vel"].params["velocity_stages"]
             if e["step"] < 10000 * 24]
assert len(_ests) == 2 and _ests[-1]["lin_vel_x"] == (-1.5, 2.0), \
    f"o envelope de velocidade não está cortado no 2º estágio: {_ests}"
assert all(e["step"] == f["step"] and e["lin_vel_x"] == f["lin_vel_x"]
           for e, f in zip(_ests, _fab_ests)), \
    f"`step` ou `lin_vel_x` dos dois primeiros estágios divergiram do molde: {_ests}"
assert _ests[0]["ang_vel_z"] == (-0.5, 0.5), \
    f"o estágio 0 tem de ficar em ±0,5 — é a rampa de uma run do zero: {_ests[0]}"
assert _ests[-1]["ang_vel_z"] == (-1.6, 1.6) == cfg.env.commands["twist"].ranges.ang_vel_z, \
    "clone anterior ao envelope de giro: o topo ainda é ±0,7 (volta em 9 s), ou a " \
    "faixa BASE não subiu junto — e é a BASE que o `exporta_cena` grava no " \
    f"`envelope_treino` do `pilota`: {_ests[-1]}, base {cfg.env.commands['twist'].ranges.ang_vel_z}"
assert "std" not in rw["track_angular_velocity"].params \
    and rw["track_angular_velocity"].params["sigma_min"] \
        == fab.rewards["track_angular_velocity"].params["std"], \
    "clone anterior ao envelope de giro: o σ do giro ainda é FIXO, e com o topo em " \
    "1,6 rad/s quem não gira recebe `exp(−1,6²/0,5) = 0,006` — kernel morto"
# ⚠ MESMO PESO do `dof_pos_limits` que ele substitui. O que muda é a FORMA: reta
# acima de 90% do curso antes, rampa exponencial por família agora. ⚠ O PESO é o mesmo
# nos dois notebooks; o que este troca é a TABELA, lá em cima.
assert rw["limite_de_junta"].weight == -1.0
assert rw["action_rate_l2"].weight == fab.rewards["action_rate_l2"].weight == -0.1, \
    "o action_rate é o do fabricante; −1,0 é do g1_poc, que tem currículo nesse peso"
assert set(fab.events) - set(ev) == {"base_com"}, "dr.body_com_offset corrompe a heap"
assert set(ev) - set(fab.events) == {"carga_caixa", "posiciona_cena", "tamanho_caixa"}, \
    f"eventos nossos: {sorted(set(ev) - set(fab.events))}"

# --- OS DISCRIMINADORES DE CLONE ANTIGO ---
assert sorted(tm) == ["caixa_largada", "fell_over", "time_out"], \
    f"terminações {sorted(tm)} — as 3 multas de mesa viraram RECOMPENSA em 01/09; " \
    "com elas aqui, o clone é anterior a isso"
assert {"palmas_em_contato", "dorso_em_contato", "impacto_da_caixa"} <= set(cfg.env.metrics), \
    "sem `impacto_da_caixa` não há como ver a política começando a JOGAR a caixa"
assert "mu" in rw["squeeze"].params, \
    "o `squeeze` ainda usa `forca_ref` fixo de 12,0 N — clone anterior ao 917bf38"
assert "sensores_palma" in rw["unload"].params, \
    "o `unload` está sem porteiro: derrubar a caixa paga 2,0/s sem mão nenhuma"
# --- OS DISCRIMINADORES DA v2.1 (auditoria de gradientes, 2026-09-04) ---
assert rw["precise_pos"].weight == 3.0 and rw["precise_pos"].params["sigma"] == 0.18, \
    f"o preciso não cobre o aceite: peso {rw['precise_pos'].weight}, "\
    f"σ {rw['precise_pos'].params['sigma']} — com σ 0,05 ele paga 0,018 no limiar do fecho"
assert "nome_do_comando" in rw["load"].params and "sensor_apoio" in rw["load"].params, \
    "clone sem `load` completo: sem ele nada paga a caixa apoiada no BOTAR (spec §2.7)"
assert "elos_que_andam" not in rw["track_linear_velocity"].params, \
    "o rastreio ainda é gateado por CONJUNTO DE ELOS: as duas esperas e o segurar-parado "\
    "pagam 4,0/s por velocidade zero forçada (P4)"
assert "pesos_manip" in cu["elo"].params, \
    "o sorteio de elo é uniforme: o REORIENTAR inerte come metade da manipulação (P8)"
# --- OS DISCRIMINADORES DE 08/09 (o sigma da tarefa, e o robo lento) ---
assert "velocidade_por_regime" in rw and rw["velocidade_por_regime"].weight < 0, \
    "o limite de velocidade de junta esta ausente, ou na forma POSITIVA — a positiva "\
    "paga 2,0/s a um robo PARADO, em todo env (piso da estatua)"
# ⚠ tabela-por-estado §4: a DOBRADIÇA `média(relu(|v|/vmax − 1)²)`, sem clamp, e o
# limite do `standing` POR FAMÍLIA de junta (14 padrões: 1,5 em tudo, punho 1,0).
import inspect
from g1_limpo import recompensas as RC
_vms = rw["velocidade_por_regime"].params["vel_max_standing"]
assert ".*" not in _vms and len(_vms) == 14 and _vms[".*wrist.*"] == 1.0 \
    and all(v == 1.5 for p, v in _vms.items() if "wrist" not in p), \
    f"clone anterior à tabela por estado: `vel_max_standing` ainda é uma entrada só ({_vms})"
assert "torch.relu(v.abs() / vmax - 1.0) ** 2" in inspect.getsource(RC.velocidade_por_regime) \
    and "max=4.0" not in inspect.getsource(RC.velocidade_por_regime), \
    "clone anterior à dobradiça: o clamp em 4,0 ainda dá derivada ZERO acima de 2× o limite"
# ⚠ tabela-por-estado §3: o gate por estado dos dez termos é o WRAPPER `PesoPorEstado`,
# que guarda o func original em `params["func"]` e a tabela em `params["tabela"]`.
# O `rastreio_por_elo` SAIU; `nome_do_comando` segue fora dos params do rastreio.
for _n in ("staged", "precise_pos", "precise_ori", "squeeze", "unload",
           "postura_ereta", "load", "track_linear_velocity",
           "track_angular_velocity", "pose"):
    assert rw[_n].func.__name__ == "PesoPorEstado" and "func" in rw[_n].params \
        and len(rw[_n].params["tabela"]) == len(CMD.ESTADOS) == 10, \
        f"clone anterior à tabela por estado: `{_n}` não passa pelo `PesoPorEstado`"
assert "nome_do_comando" not in rw["track_linear_velocity"].params, \
    "o rastreio não precisa do nome do comando: o gate é a coluna de `limpo_estado`"
assert rw["pose"].params["func"].__name__ == "PosturaPorElo", \
    "sem `PosturaPorElo` o braço sai da média em todo elo, e o `pose` do molde "\
    "vale 0,000 com derivada ZERO a 10% da faixa de junta"
assert "velocidade_de_junta" in cfg.env.metrics, \
    "sem a metrica os tres dicts de limite so recalibram com sondagem de CPU"
assert hasattr(cfg.env.commands["alvo_caixa"], "reorientar_inerte"), "clone antigo"
assert not hasattr(cfg.env.commands["alvo_caixa"], "prob_por_nivel"), \
    "clone com `prob_por_nivel`: anterior ao balanceador B/C (spec §2.5)"
assert hasattr(cfg.env.commands["alvo_caixa"], "botar_delta_topo"), \
    "clone sem `botar_delta_topo`: anterior à abertura do BOTAR na base (spec §1.4)"
# --- OS DISCRIMINADORES DE v3.2 (soltar termina, spec `g1-limpo-soltar-termina.md` §7) ---
assert "v_solta" in tm["caixa_largada"].params \
    and "dist_max" not in tm["caixa_largada"].params, \
    "clone anterior ao v3.2: `caixa_largada` ainda usa `dist_max` (distância às "\
    "palmas) em vez de `v_solta` (velocidade relativa, spec §2)"
assert {"aproxima_caixa", "renda_manipulacao"} <= set(cfg.env.metrics) \
        and cfg.env.metrics["impacto_da_caixa"].reduce == "max", \
    "sem a régua da caixa não há como ler se ela se aproxima do alvo (P9, P10)"
# --- O DISCRIMINADOR DE v3.3 (a mão gateia o BOTAR, spec `g1-limpo-mao-no-alcancar.md` §5) ---
# ⚠ A IMPRESSÃO DIGITAL NÃO PEGA ESTA MUDANÇA: ela compara só `weight`, e nenhum peso
# muda. Este assert é a ÚNICA trava contra rodar num clone pré-v3.3.
import inspect
from g1_limpo import recompensas as RC
assert "ones_like" not in inspect.getsource(RC._alcancar), \
    "clone anterior à v3.3: o `_alcancar` ainda devolve 1 constante no BOTAR — "\
    "segurar a caixa paga 3,99/s sem exigir a mão (spec §0)"
# --- O DISCRIMINADOR DE v3.4 (o BOTAR fecha sem de_pe, spec `g1-limpo-botar-fecha-e-para.md`)
assert cfg.env.commands["alvo_caixa"].sustenta_outros_s == 0.5, \
    "clone anterior à v3.4: `sustenta_outros_s` ainda é 0,3 s"
import inspect
_src = inspect.getsource(CMD.AlvoCaixaCmd._fecha_elo_corrente)
assert "apoiada[m] & de_pe[m]" not in _src, \
    "clone anterior à v3.4: o fecho do BOTAR ainda exige `de_pe` — MEDIDO, ele falha "\
    "em 99,975% dos passos que já satisfazem os outros três (spec §0)"
# --- OS DISCRIMINADORES DE v3.5 (cauda parada de pé, spec `g1-limpo-cauda-parada-de-pe.md`)
import inspect
from g1_limpo import recompensas as RC, terminacoes as TM
assert "limpo_soltou" in inspect.getsource(RC.postura_ereta), \
    "clone anterior à v3.5: `postura_ereta` não paga a pelve depois do fecho — o robô " \
    "fica agachado colhendo a anuidade (spec §0.3)"
assert "wrist" in inspect.getsource(RC.PosturaPorElo.__init__), \
    "clone anterior à v3.5: o punho ainda sai da média do `pose` durante a pega"
# ⚠ 1,00 e NÃO 0,30 (commit bcd0e84, spec `g1-limpo-punho-e-formula-obsoleta.md`): com
# 0,30 os seis punhos a ~1,6 rad somavam 139 ao expoente e `pose = exp(−6,6) = 0,0014`
# na pega — canal morto, e o punho ia ao batente. Com 1,00: `pose = 0,48`, vivo.
_std = rw["pose"].params["std_standing"]
assert any("wrist" in k and abs(v - 1.00) < 1e-9 for k, v in _std.items()), \
    f"std do punho não é 1,00 — {_std}. O 0,30 da v3.5 matava o `pose` na pega " \
    "(0,0014 de um teto de 1,0, derivada zero); clone anterior ao bcd0e84"
assert "self._command[:, ELO] != ANDAR" in inspect.getsource(CMD.AlvoCaixaCmd._aplica_espera), \
    "clone anterior à v3.5: `VALIDA` lê o elo INTERNO — `precise_pos` e `load` pagam " \
    "duas vezes na cauda (spec §0.1)"
assert "(soltou < 0.5)" not in inspect.getsource(TM.caixa_largada), \
    "clone anterior à v3.5: tirar a caixa do alvo depois do fecho não termina"
# --- O DISCRIMINADOR DO LOTE DO GIRO (spec `g1-limpo-giro-e-teclas.md` §0)
# ⚠ O wrapper `PesoPorEstado` guarda o `func` original em `params["func"]`
# (`env_cfg.py`), portanto é lá que se lê qual termo o giro chama de verdade.
assert rw["track_angular_velocity"].params["func"].__name__ == "giro_sem_gingado", \
    "clone anterior ao lote do giro: o rastreio angular ainda soma roll e pitch da " \
    "base ao erro de guinada — canal a 6,5% do máximo, sem derivada"
assert not str(LOG_ROOT).startswith(str(RAIZ)), \
    "log dentro do clone: o re-clone apaga o checkpoint"

assert cfg.agent.logger == "tensorboard", \
    "o default do mjlab é wandb, e o `leitura.py` lê events.out.tfevents"

# --------------------------- O ESTÁGIO INICIAL DO CURRÍCULO DE COMANDO
# ⚠⚠ DO ZERO O ENVELOPE É O DO ESTÁGIO 0, e não o avançado. `commands_vel`
# (`velocity/mdp/curriculums.py:99-108`) aplica TODO estágio com
# `common_step_counter >= step`, em ordem; com o contador em 0 só o estágio 0 (step 0)
# passa, e ele SOBRESCREVE a faixa base já no primeiro tique do currículo. Por isso a
# base impressa abaixo NÃO é o que o robô recebe — imprimi as duas para o dono conferir
# a primeira linha de log sem ter de adivinhar qual venceu.
# ⚠ `lin_vel_y` não aparece em estágio nenhum do molde: ela fica na base, e é a única
# dos três que a base decide de verdade.
_e0, _e1 = _ests[0], _ests[1]
_base = cfg.env.commands["twist"].ranges
_it1 = _e1["step"] // cfg.agent.num_steps_per_env
print("\ncomando no PASSO 0 (o que o robô recebe de verdade):")
print(f"  lin_vel_x = {_e0.get('lin_vel_x', _base.lin_vel_x)}")
print(f"  lin_vel_y = {_base.lin_vel_y}   (vem da base: nenhum estágio a toca)")
print(f"  ang_vel_z = {_e0.get('ang_vel_z', _base.ang_vel_z)}   <- ±0,5, o estágio 0")
print(f"  base do `twist` = x {_base.lin_vel_x}  y {_base.lin_vel_y}  "
      f"wz {_base.ang_vel_z}  (SOBRESCRITA pelo estágio 0)")
print(f"  estágio 1 entra na iteração {_it1}: "
      f"x {_e1['lin_vel_x']}  wz {_e1['ang_vel_z']}")
if cfg.agent.max_iterations > _it1:
    print(f"  ⚠ esta sessão CRUZA o estágio 1 (roda {cfg.agent.max_iterations}): a "
          f"partir da {_it1} o envelope abre, e a recompensa cai por isso — não é regressão")

lote = NUM_ENVS * cfg.agent.num_steps_per_env
print(f"\nrecompensas = {len(rw)} (esperado 29)")
print(f"CADEIAS     = {CMD.CADEIAS}   DIM = {CMD.DIM}   N_CAIXA = {OB.N_CAIXA}")
print(f"envs        = {NUM_ENVS}")
print(f"lote do PPO = {lote} transições, "
      f"minilote {lote // cfg.agent.algorithm.num_mini_batches}")
print(f"LR          = {cfg.agent.algorithm.learning_rate} ({cfg.agent.algorithm.schedule})")
print(f"log_root    = {cfg.log_root}")
print("\n⚠ O mjlab abre a pasta de run com o timestamp de agora; os pesos nascem")
print("  aleatórios. Nada aqui lê checkpoint — se você queria CONTINUAR, o notebook")
print("  é o `g1_limpo_kaggle.ipynb`.")
print("\n[CONFERE] tudo ok. começando do zero.\n")

# ⚠ A IMPRESSÃO DIGITAL DOS PESOS, E SÓ A GRAVAÇÃO. Do zero não há o que comparar: a
# trava de warm-start (ler o json antigo e acusar peso trocado) é do notebook de
# continuação e saiu daqui junto com a retomada. A gravação FICA, porque é este arquivo
# que a próxima sessão vai ler — o `empacota()` o leva no zip e a célula da API o sobe
# ao lado do `.pt`.
# ⚠ `mkdir` porque o `LOG_ROOT` nasce vazio: a pasta do experimento ainda não existe.
raiz_exp.mkdir(parents=True, exist_ok=True)
(raiz_exp / f"{RUN}.pesos.json").write_text(json.dumps(
    {k: float(v.weight) for k, v in cfg.env.rewards.items()},
    indent=1, sort_keys=True))

# ⚠ Colab: a cópia periódica para o Drive começa ANTES do lançamento. Thread daemon
# (morre com o kernel, não segura o `finally`); a cada 10 min, só o ÚLTIMO `model_*.pt`
# da run nova, para `MyDrive/g1_limpo/em_curso/`. Contra a QUEDA da sessão: a perda
# máxima passa a ser 10 min. Na Kaggle o dono escolheu a API do dataset (célula do fim).
if COLAB:
    copia_periodica_para_drive()

# ⚠ `try/finally`, e não a linha nua. Assim o pacote de saída nasce também quando o
# treino estoura (OOM é o caso provável num T4) ou quando você interrompe o kernel. O
# `empacota` e o `copia_para_drive` vêm da célula do pacote.
try:
    launch_training(g1_limpo.TASK_ID, cfg)
finally:
    empacota()
    if COLAB:
        # o zip, o `model_*.pt` final e o `.pesos.json` vão para `MyDrive/g1_limpo`.
        # Substitui a célula da API da Kaggle; a célula do fim repete isto à mão.
        copia_para_drive()

## Persistência sem commit — subir o checkpoint pela API

⚠ **Isto é o que funciona sem você presente.** O download do navegador exige a aba
aberta e um clique; isto não exige nada. É a resposta certa para "não estou rodando com
commit".

Precisa de três coisas, uma vez só:

1. **Add-ons → Secrets** → crie `KAGGLE_USERNAME` e `KAGGLE_KEY`. Os valores estão em
   <https://www.kaggle.com/settings> → API → **Create New Token** (baixa um
   `kaggle.json` com os dois campos).
2. **Internet → On.**
3. **O dataset `g1-limpo-zero` tem de EXISTIR.** O `kaggle datasets version` sobe uma
   versão nova de um dataset que já existe; ele não cria o primeiro. Crie-o uma vez,
   vazio, em <https://www.kaggle.com/datasets> → **New Dataset**, com o título
   `G1-Limpo-Zero` (o slug sai `g1-limpo-zero`) e qualquer arquivo pequeno dentro. A
   célula abaixo diz isso em português se a API recusar.

⚠⚠ **O slug é `g1-limpo-zero`, nunca `g1-limpo-v2`.** Este é o ponto em que uma troca
de nome corrompe a linhagem em silêncio — o comentário `⚠⚠` da célula explica o
mecanismo medido.

⚠ Ele sobe o **`.pt` cru**, e não o zip. É de propósito: a célula de origem do
`g1_limpo_kaggle.ipynb` procura `model_*.pt` com `rglob`, portanto um `.pt` no dataset é
achado direto na próxima sessão. Um zip obrigaria a descompactar antes.

⚠ **Uma versão nova SUBSTITUI o conteúdo do dataset.** A `zero01` antiga sai — o dono
autorizou sobrescrevê-la (2026-09-15): ela quase não rodou e não é referência. E a
Kaggle guarda as versões antigas, dá para voltar a qualquer uma.

In [ ]:
# =====================================================================
#  PERSISTE O CHECKPOINT FORA DA SESSÃO
#  Kaggle: sobe como NOVA VERSÃO DO DATASET (não precisa de commit).
#  Colab:  copia para MyDrive/g1_limpo. Redundante com o `finally` do treino, DE
#          PROPÓSITO — é a célula que você roda à mão se o `finally` não rodou.
# =====================================================================
import json, os, pathlib, shutil, subprocess

# reusa a busca do `empacota` — sem repetir a lógica e sem chance de divergir dela
if ULTIMO_CKPT is None:
    empacota(baixa=False)
assert ULTIMO_CKPT is not None, "nada para subir: o treino não salvou checkpoint"

if COLAB:
    copia_para_drive()
else:
    # ⚠⚠ `g1-limpo-zero`, E NUNCA `g1-limpo-v2`. O MOTIVO É MEDIDO e o modo de falha é
    # SILENCIOSO: a célula de origem do notebook de CONTINUAÇÃO
    # (`g1_limpo_kaggle.ipynb`) acha o `model_*.pt` de MAIOR NÚMERO por `rglob` em TODO
    # o `/kaggle/input`, sem olhar de que dataset ele veio. Se o treino do zero
    # gravasse no `g1-limpo-v2`, a sessão seguinte de continuação acharia lá o
    # `model_16000` do bloco 19 — número maior que qualquer coisa que uma run do zero
    # produza numa sessão — e continuaria a LINHAGEM ERRADA, sem uma linha de aviso.
    # Duas linhagens, dois datasets.
    SLUG = "g1-limpo-zero"      # o slug do SEU dataset, sem o nome de usuário

    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
    usuario = os.environ["KAGGLE_USERNAME"]

    # ⚠ PASTA PRÓPRIA, com SÓ o que sobe. O `kaggle datasets version` envia o diretório
    # INTEIRO — apontá-lo para /kaggle/working mandaria o clone do repo e os 30
    # checkpoints junto.
    envio = pathlib.Path("/kaggle/working/envio")
    shutil.rmtree(envio, ignore_errors=True)
    envio.mkdir()
    shutil.copy2(ULTIMO_CKPT, envio / ULTIMO_CKPT.name)
    dig = LOG_ROOT / "g1_limpo" / f"{RUN}.pesos.json"
    if dig.exists():
        shutil.copy2(dig, envio / dig.name)
    (envio / "dataset-metadata.json").write_text(json.dumps(
        {"title": "G1-Limpo-Zero", "id": f"{usuario}/{SLUG}",
         "licenses": [{"name": "CC0-1.0"}]}, indent=1))
    print("vai subir:", sorted(p.name for p in envio.iterdir()))

    r = subprocess.run(["kaggle", "datasets", "version", "-p", str(envio),
                        "-m", f"{RUN} it{ULTIMA_IT}", "-r", "skip"],
                       capture_output=True, text=True)
    saida = (r.stdout or "") + (r.stderr or "")
    print(saida)

    # ⚠ NADA DE `assert` CRU AQUI. A falha mais provável desta célula é o dataset AINDA
    # NÃO EXISTIR — `kaggle datasets version` sobe versão de um dataset que já existe, e
    # não cria o primeiro. Um traceback de AssertionError não diz o que fazer, e o dono
    # descobre isso justamente quando acabou de gastar 10 h de GPU.
    if r.returncode != 0:
        print("=" * 70)
        print("O UPLOAD NÃO ACONTECEU. O checkpoint AINDA ESTÁ EM DISCO:")
        print(f"   {ULTIMO_CKPT}")
        print("   Baixe o zip pela célula do pacote ANTES de fechar esta aba — a")
        print("   sessão morta leva /kaggle/working inteiro.")
        print("=" * 70)
        if "404" in saida or "not found" in saida.lower():
            print(f"\nCAUSA MAIS PROVÁVEL: o dataset {usuario}/{SLUG} NÃO EXISTE.")
            print("Crie-o UMA VEZ, vazio, e rode esta célula de novo:")
            print("   1. https://www.kaggle.com/datasets  ->  New Dataset")
            print(f"   2. Title: G1-Limpo-Zero   (o slug tem de sair {SLUG!r})")
            print("   3. suba qualquer arquivo pequeno (um .txt de uma linha) e Create")
            print("   4. desta vez em diante o `kaggle datasets version` desta célula")
            print("      substitui o conteúdo sozinho")
            print(f"\n⚠ NÃO aponte o SLUG para `g1-limpo-v2` para contornar isto: os")
            print("  dois datasets são linhagens diferentes, e misturá-los faz o")
            print("  notebook de continuação retomar o bloco 19 em silêncio.")
        else:
            print("\nConfira, nesta ordem:")
            print("   - os segredos KAGGLE_USERNAME e KAGGLE_KEY existem E estão")
            print("     ANEXADOS a este notebook (Add-ons -> Secrets -> o toggle de")
            print("     cada um);")
            print("   - Internet -> On;")
            print(f"   - o dataset {usuario}/{SLUG} existe e é seu.")
        raise SystemExit(
            f"upload para {usuario}/{SLUG} falhou (exit {r.returncode}) — "
            "veja as instruções acima")

    print(f"\nsubiu {ULTIMO_CKPT.name} em {usuario}/{SLUG}")
    print("na próxima sessão: o `g1_limpo_kaggle.ipynb` o acha pelo maior número")

## A próxima sessão

**Esta sessão não se repete.** Rodar este notebook de novo começa outra vez do zero, com
pesos aleatórios, e descarta o que a sessão anterior aprendeu. Quem continua a `zero01` é
o **`g1_limpo_kaggle.ipynb`**, na mesma pasta.

Para continuar, na Kaggle:

1. abra o `g1_limpo_kaggle.ipynb`;
2. **Add Input → `g1-limpo-zero`**, e **NÃO** anexe o `G1-Limpo-V2` junto. A célula de
   origem de lá acha o `model_*.pt` de MAIOR número em TODO o `/kaggle/input`: com os
   dois anexados ela pegaria o `model_16000` do bloco 19 e continuaria a linhagem
   errada, sem avisar;
3. troque o `RUN` de lá para `"zero01"`: é ele que faz o outro notebook procurar a
   pasta de run DESTA sessão, e é ele que faz a impressão digital dos pesos bater;
4. confira a tabela do `limite_de_junta`. Continuar a `zero01` com a tabela LARGA do
   knob afrouxa o freio no meio do treino e a impressão digital NÃO pega — ela compara
   `weight`, e o peso não muda. Se quiser seguir com a rampa do zero, leve o override
   desta célula de treino junto.

No Colab: o `finally` do treino (ou a célula da API) já deixou o `model_*.pt`, o
`.pesos.json` e o zip em `MyDrive/g1_limpo/`. Se a sessão CAIU, o último checkpoint está
em `MyDrive/g1_limpo/em_curso/` — a busca do outro notebook é recursiva e o acha sozinha.

**O que olhar no log desta sessão**, na ordem:

| iteração | canal | o que ele responde |
|---|---|---|
| ~400 | `Episode_Metrics/fell_over` ÷ `Episode_Reward/postura_ereta` | ele fica de pé? é o primeiro marco |
| ~1000 | `Metrics/twist/razao_marcha` | passou de 0,50? é a marcha se formando |
| ~2500 | `Curriculum/forma` (fatia de locomoção) | começou a descer de 0,95? é a manipulação entrando |
| qualquer | `Episode_Reward/limite_de_junta` | com a rampa do zero ele cobra cedo; se ficar preso no teto, a rampa está estreita demais |
| a 1ª | `Collection time` | corrija o `SEG_POR_ITER` com o valor real |
| ~5000 | `Metrics/twist/lin_vel_x_max` | o estágio 1 do envelope entra aqui; `lin_vel_x` sobe para (−1,5; 2,0) |